## Importing Necessary Libraries

In [264]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import re
import emoji
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
import string
from gensim.models import Word2Vec
from transformers import BertTokenizer, BertModel
import torch

In [265]:
nltk.download('punkt')
nltk.download('stopwords')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\ADMIN\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\ADMIN\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

## Loading Data and Investing

In [266]:
posts_path =r"C:\Users\ADMIN\Desktop\ITDSIU21099_HoangVanManh\Fashion-Marketing-Automation-Solutions\data\processed\posts.csv"
df = pd.read_csv(r"C:\Users\ADMIN\Desktop\ITDSIU21099_HoangVanManh\Fashion-Marketing-Automation-Solutions\data\processed\posts.csv")
# Drop column type and image
df = df.drop(columns=['type', 'image'])

## Data Pre-Processing

In [267]:
def overall(df):
    print ("Rows : " ,df.shape[0])
    print ("Columns : " ,df.shape[1])
    print ("\nFeatures : \n" ,df.columns.tolist())
    print ("\nMissing values : ", df.isnull().sum().values.sum())
    print ("\nUnique values : \n", df.nunique())
    
overall(df)

Rows :  827
Columns :  7

Features : 
 ['post_id', 'timestamp', 'ownerUsername', 'caption', 'hashtags', 'likesCount', 'commentsCount']

Missing values :  360

Unique values : 
 post_id          827
timestamp        827
ownerUsername     48
caption          777
hashtags         359
likesCount       787
commentsCount    358
dtype: int64


#### Check missing value (Post Dataframe)

In [268]:
df.isnull().sum()

post_id            0
timestamp          0
ownerUsername      0
caption            7
hashtags         353
likesCount         0
commentsCount      0
dtype: int64

In [269]:
# Fill missing value in caption and hashtags column by ""
df['caption'] = df['caption'].fillna('')
df['hashtags'] = df['hashtags'].fillna('')

In [270]:
df.isnull().sum()

post_id          0
timestamp        0
ownerUsername    0
caption          0
hashtags         0
likesCount       0
commentsCount    0
dtype: int64

In [271]:
df.head()

,post_id,timestamp,ownerUsername,caption,hashtags,likesCount,commentsCount
0,1,2023-06-02 16:34:43+00:00,maryleest,Cannes 2023 with @kilianparis 🤍 #KilianCannes ...,KilianCannes,32070,142
1,2,2023-01-01 07:49:16+00:00,tinaabeysekara,"As the clock struck midnight to ring in 2022, ...",,6565,86
2,3,2023-05-29 18:57:51+00:00,maryleest,The famous stairs 🤎 Photo @gustave_durin Dre...,,28936,123
3,4,2023-03-07 19:30:42+00:00,stephaniebroek,I visualized this moment so many times before....,CHANELFallWinter,4764,215
4,5,2023-05-23 21:11:12+00:00,maryleest,Got to witness such historical moment in cinem...,CannesFilmFestival,13379,123


In [272]:
# Type of each column
df.dtypes

post_id           int64
timestamp        object
ownerUsername    object
caption          object
hashtags         object
likesCount        int64
commentsCount     int64
dtype: object

### "Caption" Column

In [273]:
# Replace emoji by text
#df['caption'] = df['caption'].apply(lambda x: emoji.demojize(x) if isinstance(x, str) else x)
# Lowercase text
#df['caption'] = df['caption'].str.lower()
#punctuation_to_remove = ''.join(ch for ch in string.punctuation if ch not in [':', '_'])
#df['caption'] = df['caption'].apply(lambda x: re.sub(rf"[{re.escape(punctuation_to_remove)}]", '', x))
#df['tokens'] = df['caption'].apply(lambda x: word_tokenize(x))
#stop_words = set(stopwords.words('english'))
#df['tokens'] = df['tokens'].apply(lambda tokens: [word for word in tokens if word not in stop_words])

In [274]:
# Clean caption
def clean_caption(text):
    if pd.isna(text) or text == 'NaN':
        return ""
    text = re.sub(r'http\S+|www\S+|https\S+', '', text)
    text = re.sub(r'[^\w\s\U0001F000-\U0001F9FF]', ' ', text)
    text = re.sub(r'<.*?>', '', text)
    text = text.lower()
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text

df['clean_caption'] = df['caption'].apply(clean_caption)

In [275]:
# Process Emoji in caption
def process_emoji(text):
    emojis_list = [c for c in text if c in emoji.EMOJI_DATA]
    
    emoji_descriptions = []
    for e in emojis_list:
        emoji_name = emoji.demojize(e).replace(':', '').replace('_', ' ')
        emoji_descriptions.append(emoji_name)
    
    text_without_emoji = ''.join(c for c in text if c not in emoji.EMOJI_DATA)
    
    return text_without_emoji, emoji_descriptions

df[['caption_no_emoji', 'emoji_descriptions']] = df['clean_caption'].apply(lambda x: pd.Series(process_emoji(x)))


#### "Hashtag" column

In [276]:
# Process hashtags from hashtag column
def process_hashtags_from_column(hashtag_text):
    if pd.isna(hashtag_text) or hashtag_text == 'nan' or hashtag_text == '':
        return []
    
    if hashtag_text.startswith('[') and hashtag_text.endswith(']'):
        hashtag_text = hashtag_text[1:-1]
        
        tags = [tag.strip().strip("'").strip('"') for tag in hashtag_text.split(',')]
    else:
        tags = [tag.strip().strip('#') for tag in re.split(r'[,#]', hashtag_text) if tag.strip()]
    
    processed_hashtags = []
    for tag in tags:
        if tag:
            words = re.findall(r'[A-Z]?[a-z]+|[A-Z]+(?=[A-Z]|$)', tag)
            if not words:
                words = [tag]
            processed_hashtags.extend([word.lower() for word in words])
    
    return processed_hashtags

df['hashtags'] = df['hashtags'].apply(process_hashtags_from_column)

In [277]:
# Tokenize and remove stopwords
def tokenize_and_remove_stopwords(text):
    
    tokens = word_tokenize(text)
    
    stop_words = set(stopwords.words('english'))
    filtered_tokens = [word for word in tokens if word not in stop_words and word not in string.punctuation]
    
    return filtered_tokens

df['caption_tokens'] = df['caption_no_emoji'].apply(tokenize_and_remove_stopwords)

In [278]:
df['combined_tokens'] = df.apply(lambda row: row['caption_tokens'] + row['emoji_descriptions'] + row['hashtags'], axis=1)

In [279]:
df.head()

,post_id,timestamp,ownerUsername,caption,hashtags,likesCount,commentsCount,clean_caption,caption_no_emoji,emoji_descriptions,caption_tokens,combined_tokens
0,1,2023-06-02 16:34:43+00:00,maryleest,Cannes 2023 with @kilianparis 🤍 #KilianCannes ...,"[kilian, cannes]",32070,142,cannes 2023 with kilianparis 🤍 kiliancannes we...,cannes 2023 with kilianparis kiliancannes wea...,[white heart],"[cannes, 2023, kilianparis, kiliancannes, wear...","[cannes, 2023, kilianparis, kiliancannes, wear..."
1,2,2023-01-01 07:49:16+00:00,tinaabeysekara,"As the clock struck midnight to ring in 2022, ...",[],6565,86,as the clock struck midnight to ring in 2022 i...,as the clock struck midnight to ring in 2022 i...,"[woman dancing, medium skin tone, dizzy, mediu...","[clock, struck, midnight, ring, 2022, cried, w...","[clock, struck, midnight, ring, 2022, cried, w..."
2,3,2023-05-29 18:57:51+00:00,maryleest,The famous stairs 🤎 Photo @gustave_durin Dre...,[],28936,123,the famous stairs 🤎 photo gustave_durin dress ...,the famous stairs photo gustave_durin dress m...,[brown heart],"[famous, stairs, photo, gustave_durin, dress, ...","[famous, stairs, photo, gustave_durin, dress, ..."
3,4,2023-03-07 19:30:42+00:00,stephaniebroek,I visualized this moment so many times before....,"[chanel, fall, winter]",4764,215,i visualized this moment so many times before ...,i visualized this moment so many times before ...,"[white heart, black heart]","[visualized, moment, many, times, first, chane...","[visualized, moment, many, times, first, chane..."
4,5,2023-05-23 21:11:12+00:00,maryleest,Got to witness such historical moment in cinem...,"[cannes, film, festival]",13379,123,got to witness such historical moment in cinem...,got to witness such historical moment in cinem...,[broken heart],"[got, witness, historical, moment, cinema, kil...","[got, witness, historical, moment, cinema, kil..."


In [ ]:
df_processed = df[['post_id', 'hashtags', 'clean_caption', 'caption_no_emoji','emoji_descriptions', 'caption_tokens', 'combined_tokens']]
df_processed.head(5)

,post_id,hashtags,clean_caption,caption_no_emoji,emoji_descriptions,caption_tokens,combined_tokens
0,1,"[kilian, cannes]",cannes 2023 with kilianparis 🤍 kiliancannes we...,cannes 2023 with kilianparis kiliancannes wea...,[white heart],"[cannes, 2023, kilianparis, kiliancannes, wear...","[cannes, 2023, kilianparis, kiliancannes, wear..."
1,2,[],as the clock struck midnight to ring in 2022 i...,as the clock struck midnight to ring in 2022 i...,"[woman dancing, medium skin tone, dizzy, mediu...","[clock, struck, midnight, ring, 2022, cried, w...","[clock, struck, midnight, ring, 2022, cried, w..."
2,3,[],the famous stairs 🤎 photo gustave_durin dress ...,the famous stairs photo gustave_durin dress m...,[brown heart],"[famous, stairs, photo, gustave_durin, dress, ...","[famous, stairs, photo, gustave_durin, dress, ..."
3,4,"[chanel, fall, winter]",i visualized this moment so many times before ...,i visualized this moment so many times before ...,"[white heart, black heart]","[visualized, moment, many, times, first, chane...","[visualized, moment, many, times, first, chane..."
4,5,"[cannes, film, festival]",got to witness such historical moment in cinem...,got to witness such historical moment in cinem...,[broken heart],"[got, witness, historical, moment, cinema, kil...","[got, witness, historical, moment, cinema, kil..."
